# neural_net_walk_forward_validation_tuning

MLP validation tuning using the full-history session-aligned dataset.

The notebook tests compact neural-network hyperparameter settings and small feature-set variants, including alternative z-score windows, percentile transforms, clipped z-scores, missingness flags, and one-day lagged signal variants. Selection is based only on walk-forward validation, then the selected configurations are evaluated on the untouched test split.


In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [ ]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

MLP_PARAM_GRID = [
    {
        "param_set": "mlp_8_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (8,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-2_lr1e-3_b128",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-2,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_8_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (16, 8),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_16_alpha1e-3_lr5e-4_b128",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 5e-4,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-4_lr1e-3_b256",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_alpha1e-3_lr1e-3_b256",
        "hidden_layer_sizes": (32,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_16_alpha1e-2_lr5e-4_b256",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-2,
        "learning_rate_init": 5e-4,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
]

pd.DataFrame(MLP_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(MLP_PARAM_GRID)


In [ ]:
from __future__ import annotations



def build_mlp_pipeline_from_params(params: dict) -> Pipeline:
    mlp_params = dict(params)
    mlp_params.pop("param_set", None)
    mlp_params["hidden_layer_sizes"] = tuple(mlp_params["hidden_layer_sizes"])
    mlp_params.setdefault("random_state", CONFIG["random_state"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(**mlp_params)),
        ]
    )


In [ ]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_positive_rate": modeled_split_df["target"].mean(),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df


In [ ]:
walk_forward_fold_summary_df


In [ ]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df


In [ ]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Tune decision threshold in each validation fold: {CONFIG['tune_decision_threshold']}")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"MLP parameter sets: {len(MLP_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(MLP_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


In [ ]:
from __future__ import annotations


MLP_PARAM_COLUMNS = [
    "hidden_layer_sizes",
    "activation",
    "solver",
    "alpha",
    "learning_rate_init",
    "batch_size",
    "max_iter",
    "early_stopping",
    "n_iter_no_change",
    "tol",
]
MLP_PARAM_COLUMNS_FOR_DISPLAY = MLP_PARAM_COLUMNS


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def normalize_hidden_layer_sizes(value) -> tuple[int, ...]:
    if isinstance(value, tuple):
        return tuple(int(item) for item in value)
    if isinstance(value, list):
        return tuple(int(item) for item in value)
    if isinstance(value, np.ndarray):
        return tuple(int(item) for item in value.tolist())
    return (int(value),)


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "hidden_layer_sizes": normalize_hidden_layer_sizes(row["hidden_layer_sizes"]),
        "activation": row["activation"],
        "solver": row["solver"],
        "alpha": float(row["alpha"]),
        "learning_rate_init": float(row["learning_rate_init"]),
        "batch_size": int(row["batch_size"]),
        "max_iter": int(row["max_iter"]),
        "early_stopping": bool(row["early_stopping"]),
        "n_iter_no_change": int(row["n_iter_no_change"]),
        "tol": float(row["tol"]),
    }


def best_threshold_for_balanced_accuracy(y_true: pd.Series, scores: np.ndarray) -> tuple[float, float]:
    return ClassificationMetrics.best_threshold_for_balanced_accuracy(
        y_true,
        scores,
        min_quantile=CONFIG["threshold_min_quantile"],
        max_quantile=CONFIG["threshold_max_quantile"],
        grid_size=CONFIG["threshold_grid_size"],
        default_threshold=0.5,
    )


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    return ClassificationMetrics.metrics_from_scores(y_true, scores, threshold)


def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_mlp_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    pipeline = build_mlp_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"])
    scores = pipeline.predict_proba(eval_features_df[features])[:, 1]
    fitted_model = pipeline.named_steps["model"]

    if tune_threshold:
        decision_threshold, _ = best_threshold_for_balanced_accuracy(
            eval_features_df["target"],
            scores,
        )
    elif decision_threshold is None:
        decision_threshold = 0.5

    metric_result = metrics_from_scores(eval_features_df["target"], scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "decision_threshold": float(decision_threshold),
        **metric_result,
    }
    add_param_columns(row, params, MLP_PARAM_COLUMNS_FOR_DISPLAY)

    if not return_predictions:
        return row

    predictions_df = eval_features_df[["date", "ticker", "target"]].copy()
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def evaluate_mlp_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_mlp_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
            tune_threshold=CONFIG["tune_decision_threshold"],
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "balanced_accuracy": metric_mean,
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "f1_score": float(fold_results_df["f1_score"].mean()),
    }
    add_param_columns(summary_row, params, MLP_PARAM_COLUMNS_FOR_DISPLAY)
    return summary_row, fold_rows



In [ ]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in MLP_PARAM_GRID:
        summary_row, fold_rows = evaluate_mlp_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


In [ ]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=MLP_PARAM_COLUMNS_FOR_DISPLAY,
)

validation_best_by_feature_set_report_df


In [ ]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [ ]:
threshold_calibration_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    calibration_row = evaluate_mlp_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_threshold_calibration",
        tune_threshold=True,
    )
    threshold_calibration_rows.append(calibration_row)
    test_rows.append(
        evaluate_mlp_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_df,
            eval_input_df=test_df,
            split_name="test_walk_forward_selected_threshold",
            decision_threshold=calibration_row["decision_threshold"],
            tune_threshold=False,
        )
    )

threshold_calibration_results_df = pd.DataFrame(threshold_calibration_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_calibration_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    threshold_calibration_results_df=threshold_calibration_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)


In [ ]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=MLP_PARAM_COLUMNS_FOR_DISPLAY,
)

validation_selected_family_test_report_df
